In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

os.chdir('..')
assert os.path.basename(os.getcwd()) == "munich-accident-analysis", \
    "Need to be in the repo directory, restart the notebook and run this cell again"
os.makedirs("../data/plots", exist_ok=True)

# Accident Text Description Analysis

## Import the dataframes

In [ ]:
# Data Paths
unfall_table_path = "data/D_Unfall_RData.csv"
unfalltext_table_path = "data/D_Unfalltext_image.csv"

In [ ]:
unfall_table = pd.read_csv(unfall_table_path)
unfalltext_table = pd.read_csv(unfalltext_table_path)

In [ ]:
unfall_table.shape

In [ ]:
unfalltext_table.shape

In [ ]:
unfall_table.head(3)

In [ ]:
unfalltext_table.head(3)

## Accident text analysis

### Unique & count check

In [ ]:
# Each row has a unique UN_KEY in unfalltext table
unfalltext_table.UN_KEY.nunique()

In [ ]:
# All Text descriptions are NOT unique
print("Unique Text Count: ", unfalltext_table.Text.nunique())

In [ ]:
# Count unique descriptions and sort in descending order
description_counts = unfalltext_table['Text'].value_counts()
description_counts_df = description_counts.reset_index()
description_counts_df.columns = ['description', 'count']

In [ ]:
description_counts_df.head(3)

In [ ]:
description_counts_df.description[0]

In [ ]:
# Number of non-unique descriptions: 110
description_counts_df[description_counts_df['count'] > 1].shape

In [ ]:
# Merge the two tables on UN_KEY
unfall_text_table_merged = unfall_table.merge(unfalltext_table, on="UN_KEY", how="left")
unfall_text_table_merged.shape

In [ ]:
# Mapping dictionary for Unfalltyp
unfalltyp_mapping = {
    1: 'Fahrunfall',
    2: 'Abbiege-Unfall',
    3: 'Einbiegen/Kreuzen-Unfall',
    4: 'Überschreiten-Unfall',
    5: 'Unfall durch ruhenden Verkehr',
    6: 'Unfall im Längsverkehr',
    7: 'Sonstiger Unfall',
}

unfalltyp_mapping_eng = {
    1: 'Driving accident',
    2: 'Turning accident',
    3: 'Turning/Crossing accident',
    4: 'Crossing accident',
    5: 'Accident caused by stationary traffic',
    6: 'Accident in longitudinal traffic',
    7: 'Other accident',
}

unfall_text_table_merged['Unfalltyp_Text'] = unfall_text_table_merged['Unfalltyp'].map(unfalltyp_mapping)
unfall_text_table_merged['Unfalltyp_Text_EN'] = unfall_text_table_merged['Unfalltyp'].map(unfalltyp_mapping_eng)
unfall_text_table_merged['Unfalltyp_Text_EN_w_No'] = unfall_text_table_merged['Unfalltyp'].astype(str) + ': ' + unfall_text_table_merged['Unfalltyp_Text_EN']

In [ ]:
unfall_text_table_merged.head(3)

In [ ]:
# 2K rows have no text description
unfall_text_table_merged[unfall_text_table_merged.Text.isna()].shape

In [ ]:
# Unique unfall types
unfall_text_table_merged.Unfalltyp_Text_EN_w_No.value_counts(dropna=False)

### Check the Distribution of accident types

In [ ]:
# Define custom colors for each Unfalltyp
custom_palette = {
    '1: Driving accident': '#1f77b4',
    '2: Turning accident': '#ff7f0e',
    '3: Turning/Crossing accident': '#2ca02c',
    '4: Crossing accident': '#d62728',
    '5: Accident caused by stationary traffic': '#9467bd',
    '6: Accident in longitudinal traffic': '#8c564b',
    '7: Other accident': '#808080',
}

In [ ]:
# Determine sorted order of labels based on the combined column
x_order = sorted(
    unfall_text_table_merged['Unfalltyp_Text_EN_w_No'].unique(), 
    key=lambda x: int(x.split(':')[0])
)

# Calculate value counts and percentages
value_counts = unfall_text_table_merged['Unfalltyp_Text_EN_w_No'].value_counts()
percentages = value_counts / value_counts.sum() * 100

# Plot the histogram
plt.figure(figsize=(10, 6))
ax = sns.barplot(
    x=value_counts.index, 
    y=percentages.values, 
    order=x_order,
    palette=["#ff7f0e"] * len(value_counts)  # Apply the same color to all bars
)

# Add the percentage text on each bar
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f%%", label_type='edge', fontsize=10)

# Customize the plot
plt.title('Histogram of Accident Classes (2017 - 2022)')
plt.xlabel('Accident Class (Class Number and Description)')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

# Show the plot
plt.tight_layout()
# For paper
plt.title(None)
plt.savefig("data/plots/type_frequencies.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Group data to get counts of each Unfalltyp per year
unfall_counts = (
    unfall_text_table_merged
    .groupby(['Year', 'Unfalltyp_Text_EN_w_No'])
    .size()
    .reset_index(name='Count')
)

# Calculate total accidents per year
yearly_totals = (
    unfall_counts
    .groupby('Year')['Count']
    .sum()
    .reset_index(name='Total')
)

# Merge totals with the grouped data and calculate frequencies
unfall_counts = unfall_counts.merge(yearly_totals, on='Year')
unfall_counts['Frequency'] = unfall_counts['Count'] / unfall_counts['Total']

# Plot the normalized data
plt.figure(figsize=(12, 7))
sns.lineplot(
    data=unfall_counts, 
    x='Year', 
    y='Frequency', 
    hue='Unfalltyp_Text_EN_w_No',  # Use the combined column
    palette=custom_palette, 
    marker='o'
)

# Customize the plot
plt.title('Yearly Trend of Unfalltyp (Frequencies)', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.legend(title='Unfalltyp', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()

### Text length check

In [ ]:
# Count the number of words for each row in Text column
unfalltext_table['text_word_count'] = unfalltext_table.Text.str.split().apply(len)

In [ ]:
# Max word count is 530
# If we assume 4 tokens per word, then the max word count is around 2K tokens
unfalltext_table.text_word_count.describe()

In [ ]:
# Create a box plot (whisker plot)
plt.figure(figsize=(16, 9))
sns.boxplot(x=unfalltext_table.text_word_count, color="red", width=0.4)
plt.title("Whisker Plot for Accident Text Description Word Count", fontsize=20)
plt.xlabel("Word Count", fontsize=20)
plt.ylim(-0.5, 0.5)
plt.xticks(fontsize=16)
plt.show()

### Sascha's request: See if text length differs for accident severities

In [ ]:
beteiligte_table_path = "data/D_Beteiligte_RData.csv"
beteiligte_table = pd.read_csv(beteiligte_table_path)
mitfahrer_table_path = "data/D_Mitfahrer_RData.csv"
mitfahrer_table = pd.read_csv(mitfahrer_table_path)

In [ ]:
beteiligte_table['BT_FOLGEN'] = beteiligte_table['BT_FOLGEN'].fillna(4)
mitfahrer_table['MI_FOLGEN'] = mitfahrer_table['MI_FOLGEN'].fillna(4)

In [ ]:
# we always consider the worst outcome per accident. E.g. if someone died we assume the whole accident is severe. We're interested in whether text lengths differ here.
beteiligte_mitfahrer_table_merged = beteiligte_table.merge(mitfahrer_table, on="UN_KEY", how="left")
# min horzontally
beteiligte_mitfahrer_table_merged["min_unfallfolge"] = beteiligte_mitfahrer_table_merged[["BT_FOLGEN", "MI_FOLGEN"]].min(axis=1)
# min vertically
beteiligte_mitfahrer_table_merged["min_unfallfolge"] = beteiligte_mitfahrer_table_merged.groupby("UN_KEY")["min_unfallfolge"].transform("min")
beteiligte_mitfahrer_table_merged = beteiligte_mitfahrer_table_merged[["UN_KEY", "min_unfallfolge"]]
beteiligte_mitfahrer_unfalltext_table_merged = beteiligte_mitfahrer_table_merged.merge(unfalltext_table, on="UN_KEY", how="left")
beteiligte_mitfahrer_unfalltext_table_merged = beteiligte_mitfahrer_unfalltext_table_merged.drop_duplicates(subset=["Text"])
bt_folgen_labels = {1.0: "Deceased", 2.0: "Seriously injured", 3.0: "Lightly injured", 4.0: "Not injured"}
beteiligte_mitfahrer_unfalltext_table_merged["min_unfallfolge"] = beteiligte_mitfahrer_unfalltext_table_merged["min_unfallfolge"].replace(bt_folgen_labels)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 8), gridspec_kw={'width_ratios': [1, 3]})  

flier_props = {"marker": "o", "markersize": 5, "markerfacecolor": "black", "alpha": 0.1}

sns.boxplot(
    y=unfalltext_table.text_word_count, 
    ax=axes[0], 
    boxprops={"facecolor": "#696969"},
    medianprops={"color": "black"},
    flierprops=flier_props
)
axes[0].set_title("Overall Word Counts", fontsize=14)
axes[0].set_ylabel("Word Count")
axes[0].set_xlabel("")
axes[0].set_xticks([])

counts = beteiligte_mitfahrer_unfalltext_table_merged['min_unfallfolge'].value_counts()
sns.boxplot(
    x='min_unfallfolge', 
    y='text_word_count', 
    data=beteiligte_mitfahrer_unfalltext_table_merged, 
    ax=axes[1], 
    boxprops={"facecolor": "#D3D3D3"},
    medianprops={"color": "black"},  
    flierprops=flier_props
)

new_labels = [f"{label}\n(n={counts[label]})" for label in counts.index]
axes[1].set_xticklabels(new_labels)

axes[1].set_xlabel("Most Severe Consequence within Accident", labelpad=15)
axes[1].set_ylabel("")
axes[1].set_title("Word Counts for Different Accident Consequences", fontsize=14)

plt.tight_layout()
plt.savefig("data/plots/boxplots_cons.png", dpi=300, bbox_inches='tight')
# for paper no title
axes[0].set_title(None)
axes[1].set_title(None)
plt.savefig("data/plots/boxplots_cons_no_title.png", dpi=300, bbox_inches='tight')
plt.show()
